# Knowledge Distillation with hf_distiller
This notebook demonstrates:
1. Loading a teacher model from Hugging Face Hub
2. Creating a smaller student model
3. Preparing a toy dataset
4. Training the student using knowledge distillation
5. Visualizing training loss and logits comparison

You can replace the demo dataset with your own dataset for real training.

In [1]:
# Step 0 — Install requirements (run only once)
# !pip install --no-deps git+https://github.com/Dhiraj309/transformers_distillation.git

## Step 1 — Imports and Setup

In [2]:
import sys
import os
from transformers import AutoTokenizer, TrainingArguments
from datasets import Dataset
from transformers_distillation.models import load_teacher, load_student
from transformers_distillation import DistillTrainer
import torch

c:\Users\patil\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2 — Load Teacher Model

In [3]:
MODEL_NAME = 'google/flan-t5-small'

# Load teacher and tokenizer
teacher = load_teacher(model_name_or_path=MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Teacher model loaded:", teacher.__class__.__name__)
print("Tokenizer vocab size:", len(tokenizer))

Teacher model loaded: T5ForConditionalGeneration
Tokenizer vocab size: 32100


## Step 3 — Create Student Model
A smaller architecture for faster inference and lower memory usage.

In [4]:
student = load_student(
    model_name_or_path=MODEL_NAME,
    from_scratch=True,
    n_layers=4,
    n_heads=4,
    n_embd=256,
    is_pretrained=False
)
print("Student model created:", student.__class__.__name__)

Student model created: T5ForConditionalGeneration


## Step 4 — Prepare Dataset
Small in-memory dataset for demonstration. Replace with your own data for real training.

In [5]:
sources = [
    "Translate English to French: Hello world!",
    "Translate English to French: The quick brown fox jumps over the lazy dog.",
    "Translate English to French: Artificial intelligence is transforming industries.",
    "Translate English to French: Once upon a time, there was a curious developer.",
    "Translate English to French: PyTorch makes deep learning both fun and powerful."
]

targets = [
    "Bonjour le monde!",
    "Le renard brun rapide saute par-dessus le chien paresseux.",
    "L'intelligence artificielle transforme les industries.",
    "Il était une fois un développeur curieux.",
    "PyTorch rend l'apprentissage profond à la fois amusant et puissant."
]

dataset = Dataset.from_dict({"source": sources, "target": targets})

def tokenize(batch):
    # Tokenize encoder inputs
    model_inputs = tokenizer(
        batch["source"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Tokenize decoder targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["target"],
            max_length=128,
            truncation=True,
            padding="max_length"
        )["input_ids"]

    model_inputs["labels"] = labels
    return model_inputs


tokenized_dataset = dataset.map(tokenize, remove_columns=["source", "target"])
eval_dataset = tokenized_dataset.select(range(1))

Map:   0%|          | 0/5 [00:00<?, ? examples/s]c:\Users\patil\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\tokenization_utils_base.py:4006: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 5/5 [00:00<00:00, 68.94 examples/s]


## Step 5 — Define Training Arguments

In [6]:
training_args = TrainingArguments(
    output_dir='./student-llm',
    per_device_train_batch_size=1,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=1,
    save_steps=100,
    save_total_limit=5,
    report_to='none',
    lr_scheduler_type='cosine',
    warmup_steps=10,
)

## Step 6 — Initialize Distillation Trainer

In [7]:
trainer = DistillTrainer(
    teacher_model=teacher,
    student_model=student,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    training_args=training_args,
    kd_alpha=0.5,
    temperature=2.0,
    is_pretrained=False
)

c:\Users\patil\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers_distillation\trainer.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `DistillationTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


## Step 7 — Train Student Model
The student learns from both teacher outputs and ground truth labels.

In [8]:
# Keep track of loss for visualization
trainer_state = trainer.train()
losses = trainer_state.training_loss if hasattr(trainer_state, 'training_loss') else []

Step,Training Loss
1,10405.671900
2,10288.678700
3,10422.496100
4,10445.543000
5,10352.596700
6,10209.575200
7,10074.432600
8,10033.449200
9,10038.668900
10,10291.808600


## Step 8 — Evaluate Student Model

In [9]:
results = trainer.evaluate(eval_dataset = eval_dataset)
print('Evaluation results:', results)

Evaluation results: {'eval_loss': 8294.9560546875, 'eval_runtime': 0.637, 'eval_samples_per_second': 1.57, 'eval_steps_per_second': 1.57, 'epoch': 3.0}
